In [1]:
# CELL 1: Environment, paths, and reproducibility setup
import json
import re
import pandas as pd
import chromadb
import numpy as np
from pathlib import Path
from chromadb.utils import embedding_functions
from llama_cpp import Llama, LlamaGrammar

RESULTS_DIR = Path("../data/results")
ATTCK_DIR = Path("../data/attck")
CHROMA_DIR = Path("../data/chroma")

# Ensure the GGUF file is in your working directory
MODEL_PATH = "../models/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"

COMMUNITY_FILE = RESULTS_DIR / "community_assignments.csv"
TRIPLES_FILE = RESULTS_DIR / "community_triples.json"
STIX_FILE = ATTCK_DIR / "enterprise-attack.json"

REPORTS_FILE = RESULTS_DIR / "stage5_rag_reports.csv"
METRICS_FILE = RESULTS_DIR / "stage5_rag_metrics.json"

EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 10
RANDOM_SEED = 42
MAX_CHARS_PER_DOC = 350  # Prevents context window overflow

print("Stage 5 Environment Initialized")

Stage 5 Environment Initialized


In [2]:
# CELL 2: Load community assignments and extracted triples
community_df = pd.read_csv(COMMUNITY_FILE, low_memory=False)

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

# Filter out benign traffic for evaluation
eval_df = community_df[community_df["attck_technique_id"] != "BENIGN"].copy()
eval_cids = eval_df["community_id"].unique()[:20]  # Cap at 20 for stable evaluation
print(f"Loaded {len(eval_df)} rows | Evaluating {len(eval_cids)} communities")

Loaded 8683 rows | Evaluating 17 communities


In [3]:
# CELL 3: Initialize ChromaDB with ATT&CK STIX bundle
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name="attack_techniques", embedding_function=ef)

if collection.count() == 0:
    print("Populating ChromaDB...")
    with open(STIX_FILE, "r", encoding="utf-8") as f:
        bundle = json.load(f)
    
    docs, ids, metas = [], [], []
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern" or obj.get("revoked"):
            continue
        tid = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tid = ref.get("external_id")
                break
        if not tid: continue
        
        tactics = [p["phase_name"].replace("-", " ").title() for p in obj.get("kill_chain_phases", []) if p.get("kill_chain_name")=="mitre-attack"]
        text = f"ID: {tid}\nName: {obj.get('name')}\nTactic: {', '.join(tactics)}\nDescription: {obj.get('description', '')}"
        
        docs.append(text)
        ids.append(tid)
        metas.append({"technique_id": tid, "name": obj.get("name"), "tactic": ", ".join(tactics)})
        
    collection.add(ids=ids, documents=docs, metadatas=metas)
    print(f"Inserted {collection.count()} ATT&CK procedures")
else:
    print(f"ChromaDB ready: {collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB ready: 703 documents


In [4]:
def build_retrieval_query(cid):
    group = community_df[community_df["community_id"] == cid]
    
    # 1. Top ports
    ports = ", ".join([str(p) for p in group["Destination Port"].value_counts().head(3).index])
    
    # 2. Triples to natural sentences
    triples = community_triples.get(str(cid), [])
    sentences = []
    for t in triples[:6]:
        rel = t.get("relation", "").replace("_", " ")
        tgt = t.get("target", "").replace("_", " ")
        sentences.append(f"The traffic {rel} {tgt}.")
    semantic_summary = " ".join(sentences)
    
    # 3. Raw alert text snippet (first 2 alerts, truncated for context safety)
    raw_alerts = group["alert_text"].dropna().head(2).tolist()
    raw_snippet = " | ".join([a[:150] + "..." for a in raw_alerts])
    
    # 4. Combine (NO IDS labels, NO ground truth)
    return f"Network traffic on ports {ports}. {semantic_summary} Raw telemetry: {raw_snippet}."

In [5]:
# CELL 5: LLM initialization and strict grammar constraint
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "technique_id": {"type": "string"},
        "summary": {"type": "string"},
        "evidence": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["technique_id", "summary", "evidence", "next_step"]
}
report_grammar = LlamaGrammar.from_json_schema(json.dumps(REPORT_SCHEMA))

print("Loading Mistral-Nemo-Instruct-12B-Q5_K_M (may take 1-2 min)...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_threads=8,
    n_batch=512,
    verbose=False,
    seed=RANDOM_SEED,
    temperature=0,
    repeat_penalty=1.1
)
print("LLM & Grammar initialized.")

Loading Mistral-Nemo-Instruct-12B-Q5_K_M (may take 1-2 min)...


llama_context: n_ctx_seq (4096) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


LLM & Grammar initialized.


In [6]:
# CELL 6: Deterministic report generation with context truncation
def generate_report(query_text, retrieved_docs, retrieved_metas):
    # Truncate retrieved docs to prevent 4096 token overflow
    context_block = "\n\n".join([f"[{i+1}] {str(doc)[:MAX_CHARS_PER_DOC]}..." for i, doc in enumerate(retrieved_docs)])
    
    allowed_ids = [m.get("technique_id") for m in retrieved_metas]
    allowed_str = ", ".join(allowed_ids) if allowed_ids else "None"
    
    prompt = f"""[INST] You are a senior cybersecurity analyst.
Select the SINGLE most relevant ATT&CK technique ID from the retrieved context below.
You MUST choose one of these IDs: {allowed_str}
Return ONLY valid JSON matching the schema.

NETWORK ANALYSIS:
{query_text}

RETRIEVED ATT&CK CONTEXT:
{context_block}
[/INST]"""

    output = llm(
        prompt, max_tokens=300, temperature=0, seed=RANDOM_SEED,
        grammar=report_grammar, stop=["[/INST]"]
    )
    raw = output["choices"][0]["text"].strip()
    
    try:
        result = json.loads(raw)
        # Grounding validation: if model hallucinates an ID not in context, correct it
        gen_id = result.get("technique_id", "Unknown")
        if gen_id not in allowed_ids and gen_id != "Unknown":
            result["technique_id"] = "Unknown"
            result["evidence"] = "Generated ID not found in retrieved context."
        return result
    except json.JSONDecodeError:
        return {"technique_id": "Unknown", "summary": "Parse Error", "evidence": "N/A", "next_step": "Manual Review"}

In [7]:
# ADD BEFORE CELL 7
def calc_parent_match(gt_id, gen_id):
    if not gt_id or not gen_id or gen_id == "Unknown":
        return False
    # T1110.001 -> T1110 | T1110.003 -> T1110
    return gt_id.split('.')[0] == gen_id.split('.')[0]

In [8]:
# CELL 7: CUDA warmup and evaluation loop
print("Warming up LLM context...")
_ = llm("[INST] Warmup.[/INST]", max_tokens=4, temperature=0, seed=RANDOM_SEED, grammar=report_grammar)
print("Warmup complete.\n")

results = []
for cid in eval_cids:
    group = community_df[community_df["community_id"] == cid]
    gt = group["attck_technique_id"].mode().iloc[0]
    query = build_retrieval_query(cid)
    
    # Retrieve
    q_res = collection.query(query_texts=[query], n_results=TOP_K)
    docs = q_res["documents"][0]
    metas = q_res["metadatas"][0]
    retrieved_ids = [m.get("technique_id") for m in metas]
    
    # Generate
    report = generate_report(query, docs, metas)
    gen_id = report.get("technique_id", "Unknown")
    
    # Validate
        # Inside the loop, after generating report & validating:
    is_ground = gen_id in retrieved_ids or gen_id == "Unknown"
    match = (gen_id == gt) and is_ground
    parent_match = calc_parent_match(gt, gen_id) and is_ground  # NEW
    
    results.append({
        "community_id": cid,
        "ground_truth": gt,
        "retrieved_ids": retrieved_ids,
        "generated_technique_id": gen_id,
        "is_ground": is_ground,
        "match": match,
        "parent_match": parent_match,  # NEW
        "report_summary": report.get("summary", ""),
        "report_evidence": report.get("evidence", ""),
        "report_next_step": report.get("next_step", "")
    })

print(f"Evaluation complete for {len(results)} communities")

Warming up LLM context...
Warmup complete.

Evaluation complete for 17 communities


In [9]:
# CELL 8: Compute metrics and inspect 3 sample communities
def calc_hit_rate(results_list):
    hits = sum(1 for r in results_list if r["ground_truth"] in r["retrieved_ids"])
    return hits / max(len(results_list), 1)

metrics = {
    "n_evaluated": len(results),
    "retrieval_hit_rate": round(calc_hit_rate(results), 4),
    "grounding_rate": round(np.mean([r["is_ground"] for r in results]), 4),
    "exact_match_rate": round(np.mean([r["match"] for r in results]), 4),
    "parent_technique_match_rate": round(np.mean([r["parent_match"] for r in results]), 4)  # NEW
}

print("=== STAGE 5 METRICS ===")
print(json.dumps(metrics, indent=2))

print("\n=== SAMPLE REPORTS (First 3 Communities) ===")
for r in results[:3]:
    print(f"\nCommunity {r['community_id']} | GT: {r['ground_truth']}")
    print(f"Generated: {r['generated_technique_id']} | Grounded: {r['is_ground']} | Match: {r['match']}")
    print(f"Retrieved IDs: {r['retrieved_ids']}")
    print(f"Summary: {r['report_summary']}")
    print(f"Evidence: {r['report_evidence']}")
    print("-" * 60)

=== STAGE 5 METRICS ===
{
  "n_evaluated": 17,
  "retrieval_hit_rate": 0.1176,
  "grounding_rate": 1.0,
  "exact_match_rate": 0.0,
  "parent_technique_match_rate": 0.0
}

=== SAMPLE REPORTS (First 3 Communities) ===

Community 0 | GT: T1046
Generated: T1071.001 | Grounded: True | Match: False
Retrieved IDs: ['T1071.001', 'T1567', 'T1205', 'T1043', 'T1016.001', 'T1499.002', 'T1102.002', 'T1036.012', 'T1498', 'T1496.002']
Summary: Web Protocols (Command And Control)
Evidence: Network traffic on ports 80, 8080, 443 indicates activity port scanning activity targeting service http service, suggesting adversaries may be communicating using application layer protocols associated with web traffic to avoid detection.
------------------------------------------------------------

Community 2 | GT: T1110.001
Generated: T1043 | Grounded: True | Match: False
Retrieved IDs: ['T1071.002', 'T1043', 'T1205', 'T1071.001', 'T1016.001', 'T1498', 'T1071', 'T1029', 'T1205.002', 'T1499.001']
Summary: Adversar

In [10]:
# CELL 9: Save outputs
df = pd.DataFrame(results)
df.to_csv(REPORTS_FILE, index=False)

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"\n✅ Saved reports to {REPORTS_FILE}")
print(f"✅ Saved metrics to {METRICS_FILE}")


✅ Saved reports to ../data/results/stage5_rag_reports.csv
✅ Saved metrics to ../data/results/stage5_rag_metrics.json
